In [1]:
from pathlib import Path
from IPython.display import display
CONFIG = {
    'project_root': '/home/featurize/work/biological_prior_validation',
    'r_env': 'sciplex-rmeta',
    'seed': 20261008,
    'min_umi': 500,
    'chunk_rows': 500_000,
    'run_full_count_scan': True,  
}
ROOT = Path(CONFIG['project_root'])
assert (ROOT/'data/metadata_audit/download_manifest.json').is_file(), 'Please run notebook 01 first.'
assert (ROOT/'data/metadata_audit/metadata_audit_raw_initial.csv').is_file(), 'Please run notebook 02 first.'
print('Project:', ROOT)


Project: /home/featurize/work/biological_prior_validation


In [2]:
from pathlib import Path
import gzip,bz2,lzma,subprocess
import numpy as np
import pandas as pd
LINES=["A172","T98G","U87MG"]
DRUGS=["Vehicle","Lapatinib","Nintedanib","Trametinib","ZSTK474"]

def signature(b):
    for prefix, label in [(b'\x1f\x8b', 'gzip'), (b'BZh', 'bzip2'), (b'\xfd7zXZ\x00', 'xz'), (b'X\n', 'raw_r_serialization'), (b'A\n', 'raw_r_serialization'), (b'B\n', 'raw_r_serialization')]:
        if b.startswith(prefix):
            return label
    return 'unknown'

def check_unique(d, key):
    if key not in d:
        raise ValueError('Missing key: ' + key)
    d[key] = d[key].astype('string').str.strip().replace('', pd.NA)
    if d[key].isna().any() or d[key].duplicated().any():
        raise ValueError('Missing/duplicate key: ' + key + '; resolve provenance, never deduplicate arbitrarily')

def normalize(d):
    d = d.copy()
    if 'treatment' not in d:
        raise ValueError('No original treatment column')
    d['treatment_raw'] = d.treatment.astype('string')
    mapping = {x.lower(): x for x in DRUGS}
    mapping['dmso'] = 'Vehicle'
    d['drug'] = d.treatment_raw.str.strip().str.lower().map(mapping).astype('string')
    for f in ['cell_line', 'dose', 'replicate', 'gRNA_id', 'gene_id', 'hash_plate', 'top_oligo', 'RT_well', 'Lig_well', 'PCR_plate', 'source_object', 'library_id']:
        if f not in d:
            d[f] = pd.NA
    d['sgRNA_raw'] = d.gRNA_id
    d['target_raw'] = d.gene_id
    d['guide_call_requires_review'] = d.gRNA_id.astype('string').str.contains('c\\(|[,;|]', regex=True, na=False)
    d['dose_raw'] = d.dose
    d['dose_unit'] = 'source_method_uM_for_drugs_vehicle_0_is_label'
    d['control_label'] = np.where(d.drug.eq('Vehicle').fillna(False), 'vehicle', 'drug_or_unassigned')
    for f in ['culture_plate', 'culture_well', 'culture_batch', 'culture_date', 'transduction_id']:
        if f not in d:
            d[f] = pd.NA
    d['hash_label_well'] = d.top_oligo.astype('string').str.extract('_([A-H][0-9]+)$')[0]
    rt = pd.to_numeric(d.RT_well, errors='coerce')
    expected = pd.Series(pd.NA, index=d.index, dtype='string')
    expected.loc[rt.between(1, 192)] = 'replicate_1'
    expected.loc[rt.between(193, 384)] = 'replicate_2'
    d['expected_replicate_from_RT'] = expected
    rep = d.replicate.astype('string').str.strip()
    known = expected.notna() & rep.notna()
    d['RT_replicate_check'] = 'unknown'
    d.loc[known, 'RT_replicate_check'] = np.where(rep.loc[known].eq(expected.loc[known]), 'consistent', 'conflict')
    d['context_eligible'] = d.cell_line.isin(LINES) & d.drug.isin(DRUGS)
    return d

def count_tables(d):
    good = d.loc[d.context_eligible]
    fine = ['cell_line', 'drug', 'replicate', 'dose_raw', 'dose_unit', 'sgRNA_raw', 'target_raw', 'control_label', 'hash_plate', 'hash_label_well', 'library_id', 'culture_batch']
    counts = good.groupby(fine, dropna=False, observed=True).size().rename('n_cells').reset_index()
    assert int(counts.n_cells.sum()) == len(good)
    universe = pd.MultiIndex.from_product([LINES, DRUGS], names=['cell_line', 'drug'])
    totals = good.groupby(['cell_line', 'drug'], observed=True).size().reindex(universe, fill_value=0).rename('n_cells').reset_index()
    assert totals.n_cells.sum() == len(good)
    directions = pd.DataFrame([{'development_replicate': a, 'evaluation_replicate': b, 'status': 'planned_only_independence_unresolved'} for a, b in [('replicate_1', 'replicate_2'), ('replicate_2', 'replicate_1')]])
    return (counts, totals, directions)

def extract_rds(rds, out, r_script):
    with rds.open('rb') as f:
        outer = signature(f.read(64))
    opener = {'gzip': gzip.open, 'bzip2': bz2.open, 'xz': lzma.open}.get(outer)
    if not opener:
        raise ValueError('Unsupported outer RDS compression')
    with opener(rds, 'rb') as f:
        inner = signature(f.read(64))
    if inner == 'unknown':
        raise ValueError('Unsupported inner RDS compression')
    subprocess.run(['conda', 'run', '-n', 'sciplex-rmeta', 'Rscript', str(r_script), str(rds), str(out), outer, inner], check=True)


In [3]:
R_CODE = 'suppressPackageStartupMessages({\n  library(monocle3)\n  library(SummarizedExperiment)\n})\n\nargs <- commandArgs(trailingOnly=TRUE)\n\nif (length(args) < 4) {\n  stop("Expected arguments: infile outfile outer_format inner_format")\n}\n\ninfile <- args[[1]]\noutfile <- args[[2]]\nouter_format <- args[[3]]\ninner_format <- args[[4]]\n\nmessage("Input: ", infile)\nmessage("Outer format: ", outer_format)\nmessage("Inner format after removing outer layer: ", inner_format)\n\nif (!file.exists(infile)) {\n  stop(paste("Missing input:", infile))\n}\n\ndecompress_cmd <- function(fmt, with_file=FALSE, filename=NULL) {\n  if (fmt == "gzip") {\n    if (with_file) {\n      return(paste("gzip -dc", shQuote(filename)))\n    }\n    return("gzip -dc")\n  }\n\n  if (fmt == "bzip2") {\n    if (with_file) {\n      return(paste("bzip2 -dc", shQuote(filename)))\n    }\n    return("bzip2 -dc")\n  }\n\n  if (fmt == "xz") {\n    if (with_file) {\n      return(paste("xz -dc", shQuote(filename)))\n    }\n    return("xz -dc")\n  }\n\n  stop(paste("Unsupported compression format:", fmt))\n}\n\n# Outer file is known to be compressed.\ncmd <- decompress_cmd(\n  outer_format,\n  with_file=TRUE,\n  filename=infile\n)\n\n# If the stream after the outer layer is itself compressed,\n# remove that compression as a second stage.\nif (inner_format %in% c("gzip", "bzip2", "xz")) {\n  cmd <- paste(\n    cmd,\n    "|",\n    decompress_cmd(inner_format)\n  )\n} else if (inner_format != "raw_r_serialization") {\n  stop(\n    paste(\n      "Unsupported inner format:",\n      inner_format\n    )\n  )\n}\n\nmessage("Streaming decompression command:")\nmessage(cmd)\n\nmessage("Reading R serialization...")\n\ncon <- pipe(\n  cmd,\n  open="rb"\n)\n\nobj <- tryCatch(\n  readRDS(con),\n  finally={\n    try(close(con), silent=TRUE)\n  }\n)\n\nmessage(\n  "Top-level class: ",\n  paste(class(obj), collapse=", ")\n)\n\nextract_one <- function(x, object_name) {\n  md <- tryCatch(\n    as.data.frame(\n      SummarizedExperiment::colData(x)\n    ),\n    error=function(e) NULL\n  )\n\n  if (\n    is.null(md) &&\n    requireNamespace("Biobase", quietly=TRUE)\n  ) {\n    md <- tryCatch(\n      as.data.frame(\n        Biobase::pData(x)\n      ),\n      error=function(e) NULL\n    )\n  }\n\n  if (is.null(md)) {\n    message(\n      "Skipping object \'",\n      object_name,\n      "\'; class=",\n      paste(class(x), collapse=", ")\n    )\n    return(NULL)\n  }\n\n  bc <- rownames(md)\n\n  if (\n    is.null(bc) ||\n    length(bc) != nrow(md)\n  ) {\n    bc <- tryCatch(\n      colnames(x),\n      error=function(e) NULL\n    )\n  }\n\n  if (\n    is.null(bc) ||\n    length(bc) != nrow(md)\n  ) {\n    stop(\n      paste0(\n        "Could not recover cell barcodes for object \'",\n        object_name,\n        "\'."\n      )\n    )\n  }\n\n  md$cell_barcode <- bc\n  md$source_object <- object_name\n  rownames(md) <- NULL\n\n  md\n}\n\npieces <- list()\n\nif (\n  methods::is(obj, "SingleCellExperiment") ||\n  inherits(obj, "cell_data_set")\n) {\n  pieces[["root_object"]] <- extract_one(\n    obj,\n    "root_object"\n  )\n\n} else if (is.list(obj)) {\n\n  nm <- names(obj)\n\n  if (is.null(nm)) {\n    nm <- paste0(\n      "object_",\n      seq_along(obj)\n    )\n  }\n\n  message("Top-level list length: ", length(obj))\n  message(\n    "Top-level names: ",\n    paste(head(nm, 50), collapse=", ")\n  )\n\n  for (i in seq_along(obj)) {\n    message(\n      "Inspecting object ",\n      i,\n      "/",\n      length(obj),\n      ": ",\n      nm[[i]],\n      " | class=",\n      paste(class(obj[[i]]), collapse=", ")\n    )\n\n    tmp <- extract_one(\n      obj[[i]],\n      nm[[i]]\n    )\n\n    if (!is.null(tmp)) {\n      pieces[[nm[[i]]]] <- tmp\n    }\n  }\n\n} else {\n  stop(\n    paste(\n      "Unsupported top-level object type:",\n      paste(class(obj), collapse=", ")\n    )\n  )\n}\n\npieces <- pieces[\n  !vapply(\n    pieces,\n    is.null,\n    logical(1)\n  )\n]\n\nif (length(pieces) == 0) {\n  stop("No cell metadata could be extracted.")\n}\n\nall_cols <- unique(\n  unlist(\n    lapply(\n      pieces,\n      names\n    )\n  )\n)\n\npieces <- lapply(\n  pieces,\n  function(d) {\n    missing_cols <- setdiff(\n      all_cols,\n      names(d)\n    )\n\n    for (z in missing_cols) {\n      d[[z]] <- NA\n    }\n\n    d[\n      ,\n      all_cols,\n      drop=FALSE\n    ]\n  }\n)\n\nmeta <- do.call(\n  rbind,\n  pieces\n)\n\nmessage(\n  "Extracted rows: ",\n  format(\n    nrow(meta),\n    big.mark=","\n  )\n)\n\nmessage(\n  "Extracted columns: ",\n  ncol(meta)\n)\n\nmessage("Column names:")\nmessage(\n  paste(\n    names(meta),\n    collapse="\\n"\n  )\n)\n\nout_con <- gzfile(\n  outfile,\n  open="wt"\n)\n\nwrite.table(\n  meta,\n  out_con,\n  sep="\\t",\n  quote=FALSE,\n  row.names=FALSE,\n  col.names=TRUE,\n  na=""\n)\n\nclose(out_con)\n\nmessage("Saved: ", outfile)\n'


In [4]:
from pathlib import Path
import io,gzip,json,hashlib,subprocess,platform,sys,time
import numpy as np
import pandas as pd

def write_json(path,obj):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str));tmp.replace(path)

def hash_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def source_record(root,suffix):
    name='GSM7056149_sciPlexGxE_2_'+suffix+'.gz'
    manifest=json.loads((root/'data/metadata_audit/download_manifest.json').read_text())
    record=manifest['files'].get(name,{})
    path=root/'data/raw'/name
    if record.get('status')!='complete' or not path.is_file():raise FileNotFoundError('Missing completed download: '+name)
    if path.stat().st_size!=record.get('size_bytes'):raise ValueError('Download size changed: '+name)
    return path,record

def prepare_metadata(cfg,r_code):
    root=Path(cfg['project_root']);raw=root/'data/raw';proc=root/'data/processed';audit=root/'data/metadata_audit'
    for p in [proc,audit,root/'src',root/'results/raw',root/'configs',root/'data/splits']:p.mkdir(parents=True,exist_ok=True)
    rds,record=source_record(root,'preprocessed_cds.list.RDS')
    out=proc/'fresh_author_cell_metadata.tsv.gz';receipt=proc/'fresh_author_metadata_receipt.json'
    identity=dict(download_sha256=record['sha256'],rds_size=rds.stat().st_size,rds_mtime_ns=rds.stat().st_mtime_ns,r_code_sha256=hashlib.sha256(r_code.encode()).hexdigest())
    if out.exists():
        if not receipt.exists() or json.loads(receipt.read_text()).get('source_identity')!=identity:
            raise ValueError('Existing fresh author metadata has unknown/different source; preserve before replacing')
        if hash_file(out)!=json.loads(receipt.read_text())['metadata_sha256']:raise ValueError('Author metadata cache checksum changed')
        print('Reuse verified metadata cache:',out)
    else:
        if hash_file(rds)!=record['sha256']:raise ValueError('RDS SHA256 differs from download manifest')
        subprocess.run(['conda','run','-n',cfg['r_env'],'Rscript','-e','library(monocle3);library(SummarizedExperiment);library(Biobase)'],check=True)
        script=root/'src/fresh_extract_author_metadata.R';script.write_text(r_code)
        with rds.open('rb') as f:outer=signature(f.read(64))
        if outer!='gzip':raise ValueError('Expected outer gzip RDS')
        with gzip.open(rds,'rb') as f:inner=signature(f.read(64))
        if inner=='unknown':raise ValueError('Unrecognized inner RDS format')
        temp=out.with_name(out.name+'.part.gz')
        subprocess.run(['conda','run','-n',cfg['r_env'],'Rscript',str(script),str(rds),str(temp),outer,inner],check=True)
        check=pd.read_csv(temp,sep='\t',dtype='string',keep_default_na=False,na_values=[''])
        check_unique(check,'cell_barcode')
        temp.replace(out)
        write_json(receipt,dict(source_identity=identity,metadata_sha256=hash_file(out),n_rows=len(check)))
        del check
    author=pd.read_csv(out,sep='\t',dtype='string',keep_default_na=False,na_values=[''])
    check_unique(author,'cell_barcode')
    pd.DataFrame([dict(field=c,n_rows=len(author),n_missing=int(author[c].isna().sum()),n_unique=int(author[c].nunique()),source_file=str(out)) for c in author]).to_csv(audit/'fresh_author_field_inventory.csv',index=False)
    if 'library_id' not in author and 'sample' in author:author['library_id']=author['sample']
    d=normalize(author)
    counts,totals,directions=count_tables(d)
    counts['population_scope']='author_preprocessed_population_not_yet_raw_retained_aligned'
    counts.to_parquet(audit/'author_context_counts.parquet',index=False)
    counts.to_csv(audit/'author_context_counts.csv',index=False)
    totals.to_csv(audit/'author_context_totals.csv',index=False)
    directions.to_csv(root/'data/splits/planned_directions.csv',index=False)
    d.groupby(['replicate','RT_replicate_check'],dropna=False).size().rename('n_cells').to_csv(audit/'fresh_author_RT_check.csv')
    d.groupby(['cell_line','drug','replicate','dose_raw'],dropna=False).size().rename('n_cells').to_csv(audit/'author_dose_replicate_counts.csv')
    (root/'configs/fresh_cell_audit_config.json').write_text(json.dumps(cfg,indent=2))
    try:commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],stderr=subprocess.DEVNULL,text=True).strip()
    except (OSError,subprocess.CalledProcessError):commit='not_available'
    write_json(root/'results/raw/fresh_metadata_environment.json',dict(python=sys.version,platform=platform.platform(),numpy=np.__version__,pandas=pd.__version__,seed=cfg['seed'],git_commit=commit))
    return d,totals

class HashingReader:
    def __init__(self,f):self.f=f;self.h=hashlib.sha256();self.bytes_read=0
    def read(self,n=-1):
        b=self.f.read(n);self.h.update(b);self.bytes_read+=len(b);return b
    def seekable(self):return False


def scan_counts(path,ng,nc,expected_sha=None,chunksize=500000):
    sums=np.zeros(nc,dtype=np.int64);records=0;total=0;start=time.monotonic()
    with Path(path).open('rb') as raw:
        hashed=HashingReader(raw)
        with gzip.GzipFile(fileobj=hashed,mode='rb') as gz,io.TextIOWrapper(gz) as text:
            reader=pd.read_csv(text,sep=r'\s+',header=None,dtype=np.int64,chunksize=chunksize)
            for blockno,d in enumerate(reader,start=1):
                if d.shape[1]!=3 or d.isna().any().any():raise ValueError('Invalid count schema/NA')
                d.columns=['gene_idx','cell_idx','count']
                if (d.gene_idx<1).any() or (d.gene_idx>ng).any() or (d.cell_idx<1).any() or (d.cell_idx>nc).any() or (d['count']<0).any():
                    raise ValueError('Index/count out of bounds')
                # Chunk integer range makes the group sum provably safe before summing.
                if int(d['count'].max())>np.iinfo(np.int64).max//max(1,len(d)):
                    raise OverflowError('Count values too large for exact int64 group sum')
                per=d.groupby('cell_idx',sort=False)['count'].sum()
                idx=per.index.to_numpy(dtype=np.int64)-1;values=per.to_numpy(dtype=np.int64)
                if (values>np.iinfo(np.int64).max-sums[idx]).any():raise OverflowError('Cell UMI overflow')
                sums[idx]+=values
                records+=len(d);total+=sum(map(int,values))
                if blockno==1 or blockno%20==0:print(f'Count scan: {records:,} records; {(time.monotonic()-start)/60:.1f} min',flush=True)
        if hashed.bytes_read!=Path(path).stat().st_size:raise ValueError('Compressed file not fully consumed')
        sha=hashed.h.hexdigest()
        if expected_sha and sha!=expected_sha:raise ValueError('Raw counts SHA256 differs from download manifest')
    backcheck=int(sums.sum()) if total<=np.iinfo(np.int64).max else sum(map(int,sums))
    if backcheck!=total:raise AssertionError('UMI total back-check failed')
    return sums,dict(status='full_scan_complete',coordinate_rows=records,total_umi=total,raw_sha256=sha,
                     gene_annotation_rows=ng,candidate_barcode_rows=nc,gzip_validation='full_stream_CRC_checked',
                     duplicate_policy='additive; coordinate rows are not unique nnz')

def fresh_retained(cfg):
    root=Path(cfg['project_root']);proc=root/'data/processed';audit=root/'data/metadata_audit'
    initial=pd.read_csv(audit/'metadata_audit_raw_initial.csv',dtype='string')
    ng=int(initial.loc[initial.field.eq('raw_gene_annotation_rows'),'value'].iloc[0]);nc=int(initial.loc[initial.field.eq('raw_cell_annotation_rows'),'value'].iloc[0])
    matrix,record=source_record(root,'UMI.count.matrix')
    identity=dict(raw_manifest_sha256=record['sha256'],raw_size=matrix.stat().st_size,raw_mtime_ns=matrix.stat().st_mtime_ns,ng=ng,nc=nc)
    cache=proc/'fresh_raw_barcode_umi.npy';receipt=proc/'fresh_raw_scan_receipt.json'
    if cache.exists()!=receipt.exists():raise ValueError('Incomplete/unknown scan cache; preserve before rebuilding')
    if cache.exists() and receipt.exists():
        r=json.loads(receipt.read_text())
        if r.get('source_identity')!=identity:raise ValueError('Raw scan cache source mismatch; preserve before rebuilding')
        if hash_file(cache)!=r['cache_sha256']:raise ValueError('Raw scan cache checksum mismatch')
        sums=np.load(cache,mmap_mode='r');scan=r['scan'];print('Reuse completed full count scan cache')
    else:
        if not cfg['run_full_count_scan']:raise RuntimeError('Full raw count scan is disabled. Metadata-only stage remains valid. Set run_full_count_scan=True when ready.')
        sums,scan=scan_counts(matrix,ng,nc,record['sha256'],cfg['chunk_rows'])
        temp=cache.with_name(cache.name+'.part')
        with temp.open('wb') as f:np.save(f,sums)
        temp.replace(cache)
        write_json(receipt,dict(source_identity=identity,cache_sha256=hash_file(cache),scan=scan))
    keep=np.flatnonzero(sums>=cfg['min_umi'])
    retained=pd.DataFrame(dict(cell_idx=keep+1,total_umi=np.asarray(sums[keep])))
    retained.to_parquet(proc/'fresh_retained_cell_indices.parquet',index=False)
    ann_path,_=source_record(root,'cell.annotations.txt')
    selected=[];offset=0;wanted=retained.cell_idx.to_numpy(dtype=np.int64)
    for chunk in pd.read_csv(ann_path,sep='\t',header=None,names=['cell_barcode','library_id'],dtype='string',keep_default_na=False,chunksize=cfg['chunk_rows']):
        left=np.searchsorted(wanted,offset+1);right=np.searchsorted(wanted,offset+len(chunk),side='right')
        if right>left:
            ids=wanted[left:right];part=chunk.iloc[ids-offset-1].copy();part.insert(0,'cell_idx',ids);selected.append(part)
        offset+=len(chunk)
    if offset!=nc:raise ValueError('Raw annotation row count changed')
    selected=pd.concat(selected,ignore_index=True) if selected else pd.DataFrame(columns=['cell_idx','cell_barcode','library_id'])
    cells=retained.merge(selected,on='cell_idx',how='left',validate='one_to_one')
    check_unique(cells,'cell_barcode')
    cells=cells.sort_values('cell_idx').reset_index(drop=True);cells['matrix_row']=np.arange(len(cells))
    cells.to_parquet(proc/'fresh_retained_cell_annotations.parquet',index=False)
    write_json(root/'results/raw/fresh_count_scan_summary.json',dict(scan,min_umi=cfg['min_umi'],retained_cells=len(cells)))
    return cells

def align_fresh(root,cells,author):
    root=Path(root);audit=root/'data/metadata_audit';author=author.copy();cells=cells.copy()
    check_unique(author,'cell_barcode');check_unique(cells,'cell_barcode')
    d=cells.merge(author,on='cell_barcode',how='left',validate='one_to_one',suffixes=('_raw',''),indicator='_author_match')
    if 'library_id_raw' in d:d['library_id']=d.library_id_raw
    d=normalize(d)
    d.to_parquet(root/'data/processed/fresh_audit_cell_metadata.parquet',index=False)
    counts,totals,directions=count_tables(d);counts['population_scope']='fresh_raw_min_umi_retained_cells_matched_to_author_metadata'
    counts.to_parquet(audit/'context_counts.parquet',index=False);counts.to_csv(audit/'context_counts.csv',index=False)
    totals.to_csv(audit/'context_totals.csv',index=False)
    d.loc[d._author_match.ne('both'),['cell_idx','cell_barcode','total_umi']].to_csv(audit/'fresh_unmatched_retained_cells.csv',index=False)
    author.loc[~author.cell_barcode.isin(cells.cell_barcode),['cell_barcode','cell_line','treatment']].to_csv(audit/'author_cells_outside_raw_retained.csv',index=False)
    d.groupby(['replicate','RT_replicate_check'],dropna=False).size().rename('n_cells').to_csv(audit/'replicate_RT_check.csv')
    d.groupby(['replicate','hash_plate','library_id','PCR_plate','source_object'],dropna=False).size().rename('n_cells').to_csv(audit/'replicate_batch_counts.csv')
    summary=dict(raw_retained=len(cells),author_metadata=len(author),matched=int(d._author_match.eq('both').sum()),
                 eligible=int(d.context_eligible.sum()),unmatched=int(d._author_match.ne('both').sum()),
                 author_outside_raw_retained=int((~author.cell_barcode.isin(cells.cell_barcode)).sum()),
                 RT_conflicts=int(d.RT_replicate_check.eq('conflict').sum()),
                 biological_independence='UNRESOLVED',STRING='pending_separate_download_and_mapping')
    assert summary['matched']+summary['unmatched']==summary['raw_retained']
    assert totals.n_cells.sum()==summary['eligible']
    write_json(root/'results/raw/fresh_alignment_summary.json',summary)
    return summary,totals


In [5]:
author_metadata, author_totals = prepare_metadata(CONFIG, R_CODE)
print('Author preprocessed metadata row count：', len(author_metadata))
display(author_totals)
display(pd.read_csv(ROOT/'data/metadata_audit/fresh_author_RT_check.csv'))
display(pd.read_csv(ROOT/'data/metadata_audit/author_dose_replicate_counts.csv'))


Reuse verified metadata cache: /home/featurize/work/biological_prior_validation/data/processed/fresh_author_cell_metadata.tsv.gz
Author preprocessed metadata row count： 989299


,cell_line,drug,n_cells
0,A172,Vehicle,65651
1,A172,Lapatinib,95319
2,A172,Nintedanib,75380
3,A172,Trametinib,33362
4,A172,ZSTK474,65376
5,T98G,Vehicle,70400
6,T98G,Lapatinib,102647
7,T98G,Nintedanib,49071
8,T98G,Trametinib,50865
9,T98G,ZSTK474,52472


,replicate,RT_replicate_check,n_cells
0,replicate_1,consistent,490393
1,replicate_2,consistent,498906


,cell_line,drug,replicate,dose_raw,n_cells
0,A172,Lapatinib,replicate_1,1,20763
1,A172,Lapatinib,replicate_1,10,26495
2,A172,Lapatinib,replicate_2,1,20114
3,A172,Lapatinib,replicate_2,10,27947
4,A172,Nintedanib,replicate_1,1,19805
5,A172,Nintedanib,replicate_1,10,16502
6,A172,Nintedanib,replicate_2,1,21133
7,A172,Nintedanib,replicate_2,10,17940
8,A172,Trametinib,replicate_1,1,7276
9,A172,Trametinib,replicate_1,10,6447


In [6]:
if CONFIG['run_full_count_scan']:
    retained_cells = fresh_retained(CONFIG)
    alignment_summary, raw_totals = align_fresh(ROOT, retained_cells, author_metadata)
    print(json.dumps(alignment_summary, ensure_ascii=False, indent=2))
    display(raw_totals)
    display(pd.read_csv(ROOT/'data/metadata_audit/replicate_RT_check.csv'))
else:
    print('This run only performs metadata verification by the author; the number of cells with the original UMI retained and the alignment results still need a full scan.')


Reuse completed full count scan cache
{
  "raw_retained": 1052205,
  "author_metadata": 989299,
  "matched": 989299,
  "eligible": 989299,
  "unmatched": 62906,
  "author_outside_raw_retained": 0,
  "RT_conflicts": 0,
  "biological_independence": "UNRESOLVED",
  "STRING": "pending_separate_download_and_mapping"
}


,cell_line,drug,n_cells
0,A172,Vehicle,65651
1,A172,Lapatinib,95319
2,A172,Nintedanib,75380
3,A172,Trametinib,33362
4,A172,ZSTK474,65376
5,T98G,Vehicle,70400
6,T98G,Lapatinib,102647
7,T98G,Nintedanib,49071
8,T98G,Trametinib,50865
9,T98G,ZSTK474,52472


,replicate,RT_replicate_check,n_cells
0,replicate_1,consistent,490393
1,replicate_2,consistent,498906
2,NaN,unknown,62906


In [7]:
status = [
    {'field':'author_preprocessed_metadata_rows','value':len(author_metadata),'scope':'fresh_RDS_metadata'},
    {'field':'author_context_eligible_cells','value':int(author_totals.n_cells.sum()),'scope':'fresh_RDS_metadata'},
    {'field':'author_RT_conflicts','value':int(author_metadata.RT_replicate_check.eq('conflict').sum()),'scope':'fresh_RDS_metadata'},
    {'field':'biological_replicate_independence','value':'UNRESOLVED','scope':'requires_culture_or_transduction_provenance'},
    {'field':'STRING_mapping','value':'PENDING','scope':'outside_this_notebook'},
]
if CONFIG['run_full_count_scan']:
    status += [{'field':k,'value':v,'scope':'fresh_raw_UMI_retained_alignment'} for k,v in alignment_summary.items()]
else:
    status.append({'field':'raw_retained_alignment','value':'NOT_RUN_THIS_SESSION','scope':'full_count_scan_disabled'})
status = pd.DataFrame(status)
status.to_csv(ROOT/'data/metadata_audit/metadata_audit_fresh.csv',index=False)
display(status)


,field,value,scope
0,author_preprocessed_metadata_rows,989299,fresh_RDS_metadata
1,author_context_eligible_cells,989299,fresh_RDS_metadata
2,author_RT_conflicts,0,fresh_RDS_metadata
3,biological_replicate_independence,UNRESOLVED,requires_culture_or_transduction_provenance
4,STRING_mapping,PENDING,outside_this_notebook
5,raw_retained,1052205,fresh_raw_UMI_retained_alignment
6,author_metadata,989299,fresh_raw_UMI_retained_alignment
7,matched,989299,fresh_raw_UMI_retained_alignment
8,eligible,989299,fresh_raw_UMI_retained_alignment
9,unmatched,62906,fresh_raw_UMI_retained_alignment
